# Stage 04: Data Acquisition and Ingestion

This notebook creates a reproducible ingestion workflow for the S&P 500 volatility project. It downloads daily S&P 500 prices from a public API and scrapes a small S&P 500 constituent table from a public webpage.

## Sources, Parameters, and Reproducibility

- **API source:** Yahoo Finance chart endpoint: `https://query1.finance.yahoo.com/v8/finance/chart/%5EGSPC`
- **Scraped source:** Wikipedia: `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`
- **Parameters:** ticker, start date, end date, and timeout are read from `.env`.
- **Output naming:** each raw snapshot includes its source, dataset, and UTC retrieval time.
- **Secrets:** the public API needs no key; `.env` is ignored by Git and only `.env.example` is committed.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
import requests

REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'homework' / 'homework04').exists()
)
HOMEWORK_DIR = REPO_ROOT / 'homework' / 'homework04'
RAW_DIR = HOMEWORK_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(HOMEWORK_DIR / '.env')
TICKER = os.environ.get('MARKET_TICKER', '^GSPC')
START_DATE = os.environ.get('API_LOOKBACK_START', '2025-01-01')
END_DATE = os.environ.get('API_LOOKBACK_END', '2025-02-01')
TIMEOUT_SECONDS = int(os.environ.get('REQUEST_TIMEOUT_SECONDS', '30'))
REQUEST_HEADERS = {'User-Agent': 'bootcamp-data-ingestion/1.0 (educational project)'}

print({'ticker': TICKER, 'start_date': START_DATE, 'end_date': END_DATE, 'timeout_seconds': TIMEOUT_SECONDS})

{'ticker': '^GSPC', 'start_date': '2025-01-01', 'end_date': '2025-02-01', 'timeout_seconds': 30}


/Users/yifang/bootcamp_Yifang_Qiu/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Validation Helper
The helper checks for required columns, empty files, missing values, and basic numeric or text rules before any raw file is saved.

In [2]:
def validate_dataframe(dataframe, required_columns, numeric_columns=(), text_columns=()):
    """Validate required fields, missing values, and simple column rules."""
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')
    if dataframe.empty:
        raise ValueError('The ingestion returned no rows.')

    na_counts = dataframe[required_columns].isna().sum().to_dict()
    invalid_numeric = {
        column: int((~pd.to_numeric(dataframe[column], errors='coerce').notna()).sum())
        for column in numeric_columns
    }
    blank_text = {
        column: int(dataframe[column].astype(str).str.strip().eq('').sum())
        for column in text_columns
    }
    if any(na_counts.values()) or any(invalid_numeric.values()) or any(blank_text.values()):
        raise ValueError({
            'na_counts': na_counts,
            'invalid_numeric': invalid_numeric,
            'blank_text': blank_text,
        })

    return {'shape': dataframe.shape, 'na_counts': na_counts, 'status': 'passed'}

## 1. API Pull: S&P 500 Daily Prices
The Yahoo endpoint returns JSON. The code converts timestamps and numeric price fields into a pandas DataFrame, validates the result, and saves a raw CSV snapshot.

In [3]:
api_url = f'https://query1.finance.yahoo.com/v8/finance/chart/{TICKER}'
api_params = {
    'period1': int(pd.Timestamp(START_DATE, tz='UTC').timestamp()),
    'period2': int(pd.Timestamp(END_DATE, tz='UTC').timestamp()),
    'interval': '1d',
    'events': 'history',
}

api_response = requests.get(api_url, params=api_params, headers=REQUEST_HEADERS, timeout=TIMEOUT_SECONDS)
api_response.raise_for_status()
api_payload = api_response.json()
api_result = api_payload['chart']['result'][0]

price_data = pd.DataFrame(api_result['indicators']['quote'][0])
price_data.insert(0, 'date', pd.to_datetime(api_result['timestamp'], unit='s', utc=True).tz_localize(None))
price_data['ticker'] = TICKER
price_data = price_data[['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
price_data = price_data.dropna(subset=['close']).sort_values('date').reset_index(drop=True)

api_validation = validate_dataframe(
    price_data,
    required_columns=['date', 'ticker', 'open', 'high', 'low', 'close', 'volume'],
    numeric_columns=['open', 'high', 'low', 'close', 'volume'],
    text_columns=['ticker'],
)
api_validation

{'shape': (20, 7),
 'na_counts': {'date': 0,
  'ticker': 0,
  'open': 0,
  'high': 0,
  'low': 0,
  'close': 0,
  'volume': 0},
 'status': 'passed'}

In [4]:
retrieved_at = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')
ticker_slug = TICKER.replace('^', '').replace('-', '_')
api_output_path = RAW_DIR / f'api_yahoo-finance_{ticker_slug}_{retrieved_at}.csv'
price_data.to_csv(api_output_path, index=False)
print(f'Saved {len(price_data)} API rows to {api_output_path.name}')
price_data.head()

Saved 20 API rows to api_yahoo-finance_GSPC_20260820-0454.csv


,date,ticker,open,high,low,close,volume
0,2025-01-02 14:30:00,^GSPC,5903.259766,5935.089844,5829.529785,5868.549805,3621680000
1,2025-01-03 14:30:00,^GSPC,5891.069824,5949.339844,5888.660156,5942.470215,3667340000
2,2025-01-06 14:30:00,^GSPC,5982.810059,6021.040039,5960.009766,5975.379883,4940120000
3,2025-01-07 14:30:00,^GSPC,5993.259766,6000.680176,5890.680176,5909.029785,4517330000
4,2025-01-08 14:30:00,^GSPC,5910.660156,5927.890137,5874.779785,5918.250000,4441740000


## 2. Scrape a Small Table: S&P 500 Constituents
Wikipedia publishes a public constituents table. The code first tries the table's stable `#constituents` selector and falls back to a general sortable table selector if the page markup changes.

In [5]:
scrape_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
scrape_response = requests.get(scrape_url, headers=REQUEST_HEADERS, timeout=TIMEOUT_SECONDS)
scrape_response.raise_for_status()
soup = BeautifulSoup(scrape_response.text, 'html.parser')

constituent_table = soup.select_one('table#constituents') or soup.select_one('table.wikitable.sortable')
if constituent_table is None:
    raise ValueError('Could not find the S&P 500 constituents table.')

rows = []
for row in constituent_table.select('tbody tr'):
    cells = row.find_all('td')
    if len(cells) >= 4:
        rows.append({
            'symbol': cells[0].get_text(' ', strip=True),
            'security': cells[1].get_text(' ', strip=True),
            'gics_sector': cells[2].get_text(' ', strip=True),
            'gics_sub_industry': cells[3].get_text(' ', strip=True),
        })

constituents = pd.DataFrame(rows).head(10)
scrape_validation = validate_dataframe(
    constituents,
    required_columns=['symbol', 'security', 'gics_sector', 'gics_sub_industry'],
    text_columns=['symbol', 'security', 'gics_sector', 'gics_sub_industry'],
)
scrape_validation

{'shape': (10, 4),
 'na_counts': {'symbol': 0,
  'security': 0,
  'gics_sector': 0,
  'gics_sub_industry': 0},
 'status': 'passed'}

In [6]:
scrape_output_path = RAW_DIR / f'scrape_wikipedia_sp500-constituents_{retrieved_at}.csv'
constituents.to_csv(scrape_output_path, index=False)
print(f'Saved {len(constituents)} scraped rows to {scrape_output_path.name}')
constituents

Saved 10 scraped rows to scrape_wikipedia_sp500-constituents_20260820-0454.csv


,symbol,security,gics_sector,gics_sub_industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services
5,ADBE,Adobe Inc.,Information Technology,Application Software
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders
8,AFL,Aflac,Financials,Life & Health Insurance
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services


## Assumptions and Risks

- Yahoo Finance and Wikipedia are public, third-party sources whose formats, coverage, and availability can change.
- The daily prices are raw snapshots and may differ from adjusted or licensed vendor data.
- Constituents change over time; this small scraped table is a snapshot, not historical membership data.
- Validation detects missing fields and simple type issues but does not establish financial accuracy.
- Later stages should align trading dates, manage revisions, and document any production data licenses.